# ATP 2-01.3 — Fine-Tuned LLM (Google Colab / T4 GPU)

Fine-tune a language model on ATP 2-01.3 (Intelligence Preparation of the Battlefield)
to serve as a doctrine assistant for military intelligence questions.

**Hardware:** Google Colab free tier (T4 GPU)  
**Model:** Gemma 2-2B-IT (unsloth/gemma-2-2b-it)  
**Framework:** Unsloth + QLoRA  

```
PDF → Chunks → QA Pairs → Format → Train → Evaluate → Export GGUF
  1       2        3         4        5         6           7
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Accept the [Gemma 2 license](https://huggingface.co/google/gemma-2-2b-it) on HuggingFace
3. Have a HuggingFace **read token** ready (huggingface.co/settings/tokens)
4. Have `ATP_2-01.3.pdf` ready to upload

## Step 0 — GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — Runtime → Change runtime type → T4 GPU"
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl==0.8.6" peft accelerate bitsandbytes
!pip install pdfplumber rouge-score matplotlib datasets
!pip install --upgrade "huggingface_hub[cli]"

## Step 2 — Mount Drive & Upload PDF

In [ ]:
from google.colab import drive, files
import os

drive.mount('/content/drive')

# ── Path constants ────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
PDF_PATH     = '/content/ATP_2-01.3.pdf'
CHUNKS_PATH  = f'{DRIVE_BASE}/data/chunks.jsonl'
SEEDS_PATH   = f'{DRIVE_BASE}/data/seeds.jsonl'
TRAIN_PATH   = f'{DRIVE_BASE}/data/train.jsonl'
VALID_PATH   = f'{DRIVE_BASE}/data/val.jsonl'
ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
EVAL_PATH    = f'{DRIVE_BASE}/eval/results/atp_gemma2_colab.json'
CHART_PATH   = f'{DRIVE_BASE}/eval/score_chart.png'
GGUF_DIR     = f'{DRIVE_BASE}/burns'
MODEL_NAME   = 'unsloth/gemma-2-2b-it'

for d in [f'{DRIVE_BASE}/data', f'{DRIVE_BASE}/outputs',
          f'{DRIVE_BASE}/eval/results', GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

# Upload PDF
print("Upload ATP_2-01.3.pdf when prompted:")
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(PDF_PATH, 'wb') as f:
        f.write(data)
    print(f"Saved → {PDF_PATH} ({len(data):,} bytes)")

## Step 3 — HuggingFace Login

In [ ]:
from huggingface_hub import login
login()  # Paste your HF read token when prompted

## Step 4 — Chunk PDF

Parses ATP 2-01.3 into paragraph-level chunks with chapter/appendix metadata.
Output saved to Drive so this step can be skipped on resume.

In [ ]:
import json, re, os
from pathlib import Path
import pdfplumber

ATP_CHAPTERS = {
    1: 'IPB Fundamentals',
    2: 'IPB Support to Planning',
    3: 'Step 1 - Define the Operational Environment',
    4: 'Step 2 - Describe Environmental Effects',
    5: 'Step 3 - Evaluate the Threat',
    6: 'Step 4 - Determine Threat COAs',
    7: 'Unified Action and Unique Environments',
    8: 'Multi-Domain Considerations',
}

PARA_ID_RE   = re.compile(r'^(\d+)-(\d+)\.\s+', re.MULTILINE)
APPEND_ID_RE = re.compile(r'^([A-D])-(\d+)\.\s+', re.MULTILINE)

def extract_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            pages.append(text)
    return '\n'.join(pages)

def detect_chapter(para_id):
    m = PARA_ID_RE.match(para_id + '. ')
    if m:
        ch = int(m.group(1))
        return ch, ATP_CHAPTERS.get(ch, f'Chapter {ch}')
    return None, 'Appendix'

def chunk_text(text):
    chunks = []
    chunk_id = 0
    # Split on paragraph IDs
    parts = re.split(r'(?=^(?:\d+-\d+|[A-D]-\d+)\.\s)', text, flags=re.MULTILINE)
    for part in parts:
        part = part.strip()
        if not part or len(part.split()) < 10:
            continue
        # Extract para_id from start of chunk
        m = re.match(r'^(\d+-\d+|[A-D]-\d+)\.\s+', part)
        para_id = m.group(1) if m else f'unk-{chunk_id}'
        ch_num, ch_title = detect_chapter(para_id)
        chunks.append({
            'chunk_id':    f'chunk_{chunk_id:04d}',
            'para_id':     para_id,
            'text':        part,
            'chapter_num': ch_num,
            'chapter_title': ch_title,
            'word_count':  len(part.split()),
        })
        chunk_id += 1
    return chunks

if os.path.exists(CHUNKS_PATH):
    print(f'Chunks already exist at {CHUNKS_PATH} — skipping. Delete to re-chunk.')
else:
    print('Extracting text from PDF...')
    text   = extract_text(PDF_PATH)
    chunks = chunk_text(text)
    with open(CHUNKS_PATH, 'w') as f:
        for c in chunks:
            f.write(json.dumps(c) + '\n')
    print(f'Saved {len(chunks)} chunks → {CHUNKS_PATH}')

chunks = [json.loads(l) for l in open(CHUNKS_PATH)]
print(f'Total chunks: {len(chunks)}')
print('\nSample:')
print(json.dumps(chunks[0], indent=2)[:400])

## Step 5 — Generate QA Pairs

Uses the model itself to generate QA pairs from each chunk.
Appends to Drive so the run is safely resumable after disconnects.

> **Note:** Targeting 300 pairs for Colab (vs 5,000 for the full MLX run). Adjust `TARGET` as needed.

In [ ]:
import json, random, os, gc
import torch
from unsloth import FastLanguageModel

TARGET     = 300
BATCH_SIZE = 5

QUESTION_TYPES = [
    'factual', 'procedural', 'conceptual', 'application',
    'comparative', 'scenario', 'definition', 'relationship',
]

GEN_SYSTEM = (
    "You are a military doctrine expert generating training data for ATP 2-01.3 "
    "(Intelligence Preparation of the Battlefield). Generate a QA pair from the "
    "given doctrine text. The answer must end with [Reference: ATP 2-01.3, para X-Y]."
)

def make_gen_prompt(chunk_text, q_type):
    return (
        f"<start_of_turn>user\n"
        f"{GEN_SYSTEM}\n\n"
        f"Doctrine text:\n{chunk_text[:600]}\n\n"
        f"Generate a {q_type} question and answer about this doctrine. "
        f"Format:\nQUESTION: <question>\nANSWER: <answer>\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )

def parse_qa(text, chunk, q_type, qa_id):
    q_match = re.search(r'QUESTION:\s*(.+?)(?=ANSWER:|$)', text, re.DOTALL)
    a_match = re.search(r'ANSWER:\s*(.+)', text, re.DOTALL)
    if not q_match or not a_match:
        return None
    q = q_match.group(1).strip()
    a = a_match.group(1).strip()
    if len(q) < 10 or len(a) < 20:
        return None
    return {
        'qa_id':         qa_id,
        'source_chunks': [chunk['para_id']],
        'question_type': q_type,
        'question':      q,
        'answer':        a,
        'metadata': {
            'chapter_num':   chunk.get('chapter_num'),
            'chapter_title': chunk.get('chapter_title'),
        }
    }

# Count existing seeds
existing = 0
if os.path.exists(SEEDS_PATH):
    existing = sum(1 for _ in open(SEEDS_PATH))
print(f'Existing seeds: {existing} / {TARGET}')

if existing >= TARGET:
    print('Target reached — skipping generation.')
else:
    print('Loading model for generation...')
    gen_model, gen_tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(gen_model)

    chunks_loaded = [json.loads(l) for l in open(CHUNKS_PATH)]
    random.shuffle(chunks_loaded)
    qa_count = existing

    with open(SEEDS_PATH, 'a') as out_f:
        for chunk in chunks_loaded:
            if qa_count >= TARGET:
                break
            if len(chunk['text'].split()) < 30:
                continue
            q_type = random.choice(QUESTION_TYPES)
            prompt = make_gen_prompt(chunk['text'], q_type)
            inputs = gen_tok(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                out = gen_model.generate(
                    **inputs, max_new_tokens=300, temperature=0.7,
                    do_sample=True, pad_token_id=gen_tok.eos_token_id,
                )
            decoded = gen_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            seed = parse_qa(decoded, chunk, q_type, f'qa_{qa_count:04d}')
            if seed:
                out_f.write(json.dumps(seed) + '\n')
                qa_count += 1
                if qa_count % 20 == 0:
                    print(f'  Generated {qa_count}/{TARGET}...')

    del gen_model, gen_tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Done — {qa_count} seeds saved to {SEEDS_PATH}')

seeds = [json.loads(l) for l in open(SEEDS_PATH)]
from collections import Counter
qtypes = Counter(s.get('question_type') for s in seeds)
print(f'\nTotal QA pairs: {len(seeds)}')
for qt, cnt in sorted(qtypes.items(), key=lambda x: -x[1]):
    print(f'  {qt:<20}: {cnt}')

## Step 6 — Format Training Data

Converts QA pairs into Gemma 2 chat-template format. Splits 90/10 train/val.

In [ ]:
import json, random

VAL_RATIO = 0.10
random.seed(42)

SYSTEM_PROMPT = (
    "You are a doctrine-grounded military intelligence assistant specializing in "
    "ATP 2-01.3 (Intelligence Preparation of the Battlefield, March 2019). "
    "Provide accurate, doctrinally grounded answers. "
    "Place citations at the END in [Reference: ATP 2-01.3, para X-Y] format."
)

def format_example(seed):
    q   = seed.get('question', '').strip()
    ans = seed.get('answer',   '').strip()
    if not q or not ans:
        return None
    text = (
        f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{q}<end_of_turn>\n"
        f"<start_of_turn>model\n{ans}<end_of_turn>"
    )
    return {'text': text, 'question_type': seed.get('question_type'),
            'chapter': seed.get('metadata', {}).get('chapter_title')}

seeds = [json.loads(l) for l in open(SEEDS_PATH)]
examples = [e for s in seeds if (e := format_example(s)) is not None]
random.shuffle(examples)

n_val   = max(1, int(len(examples) * VAL_RATIO))
n_train = len(examples) - n_val
train_ex = examples[:n_train]
val_ex   = examples[n_train:]

with open(TRAIN_PATH, 'w') as f:
    for ex in train_ex: f.write(json.dumps(ex) + '\n')
with open(VALID_PATH, 'w') as f:
    for ex in val_ex:   f.write(json.dumps(ex) + '\n')

print(f'Train: {n_train} | Val: {n_val}')
print('\nFormatted example (first 500 chars):')
print(train_ex[0]['text'][:500])

## Step 7 — Train (Unsloth QLoRA)

Fine-tunes Gemma 2-2B with QLoRA. Checkpoints saved to Drive.

Expected time on T4: ~40-60 minutes for 300 examples / 2 epochs.

In [ ]:
import json, os
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LENGTH = 2048
CKPT_DIR       = f'{DRIVE_BASE}/outputs/checkpoints'

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)

# Apply QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42, use_rslora=True,
)
model.print_trainable_parameters()

# Load datasets
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            obj = json.loads(line)
            if 'text' in obj:
                rows.append({'text': obj['text']})
    return Dataset.from_list(rows)

train_ds = load_jsonl(TRAIN_PATH)
val_ds   = load_jsonl(VALID_PATH)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=CKPT_DIR,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        optim='adamw_8bit',
        weight_decay=0.01,
        bf16=False, fp16=True,
        logging_steps=10,
        eval_strategy='steps', eval_steps=50,
        save_strategy='epoch',
        load_best_model_at_end=False,
        seed=42, report_to='none',
        packing=True, dataset_num_proc=2,
    ),
)

stats = trainer.train()
print(f"\nTrain loss: {stats.metrics.get('train_loss', 'N/A'):.4f}")

# Save adapter to Drive
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved → {ADAPTER_PATH}')

## Step 8 — Evaluate

Runs 24 doctrine questions against the fine-tuned model.
Scores keyword coverage (0.0-1.0). Also evaluates the base model for comparison.

In [ ]:
import gc, re, json, os
import torch
import numpy as np
from unsloth import FastLanguageModel

# Path fallback for runtime restarts
if 'DRIVE_BASE' not in dir():
    DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
    ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
    EVAL_PATH    = f'{DRIVE_BASE}/eval/results/atp_gemma2_colab.json'
    MODEL_NAME   = 'unsloth/gemma-2-2b-it'

EVAL_SYSTEM = (
    "You are a doctrine-grounded military intelligence assistant specializing in "
    "ATP 2-01.3 (Intelligence Preparation of the Battlefield, March 2019). "
    "Provide accurate, doctrinally grounded answers. "
    "Place citations at the END in [Reference: ATP 2-01.3, para X-Y] format."
)

EVAL_QUESTIONS = [
    {"q": "What are the four steps of IPB?",
     "keywords": ["define", "describe", "evaluate", "determine", "operational environment"]},
    {"q": "What is the purpose of Intelligence Preparation of the Battlefield?",
     "keywords": ["commander", "decision", "threat", "environment", "intelligence"]},
    {"q": "What does Step 1 of IPB involve?",
     "keywords": ["define", "operational environment", "area", "interest", "influence"]},
    {"q": "What is a named area of interest (NAI)?",
     "keywords": ["named", "area", "interest", "information", "activity"]},
    {"q": "What is a threat course of action (COA)?",
     "keywords": ["threat", "course", "action", "objective", "scheme"]},
    {"q": "How does IPB support the military decision-making process?",
     "keywords": ["planning", "commander", "decision", "intelligence", "staff"]},
    {"q": "What is an event template?",
     "keywords": ["event", "template", "named area", "activity", "threat"]},
    {"q": "What is the difference between the area of interest and area of influence?",
     "keywords": ["area of interest", "area of influence", "commander", "operations"]},
    {"q": "What is MCOO and what does it depict?",
     "keywords": ["modified combined", "obstacle overlay", "terrain", "movement", "trafficability"]},
    {"q": "What role does weather play in Step 2 of IPB?",
     "keywords": ["weather", "effects", "operations", "visibility", "terrain"]},
    {"q": "What is high-value target (HVT) analysis?",
     "keywords": ["high-value", "target", "threat", "capability", "course of action"]},
    {"q": "What is a decision support template (DST)?",
     "keywords": ["decision", "support", "template", "trigger", "commander"]},
    {"q": "How does IPB support targeting?",
     "keywords": ["targeting", "high-value", "high-payoff", "threat", "intelligence"]},
    {"q": "What is an intelligence gap?",
     "keywords": ["intelligence", "gap", "information", "required", "unknown"]},
    {"q": "What is a situation template?",
     "keywords": ["situation", "template", "threat", "course of action", "depict"]},
    {"q": "How does IPB address multi-domain operations?",
     "keywords": ["multi-domain", "cyber", "space", "electromagnetic", "operations"]},
    {"q": "What is the threat evaluation step of IPB?",
     "keywords": ["evaluate", "threat", "doctrine", "tactics", "capabilities"]},
    {"q": "What are priority intelligence requirements (PIR)?",
     "keywords": ["priority", "intelligence", "requirements", "commander", "decision"]},
    {"q": "What is terrain analysis in the context of IPB?",
     "keywords": ["terrain", "observation", "fields of fire", "cover", "avenues"]},
    {"q": "What is OAKOC?",
     "keywords": ["observation", "avenues", "key terrain", "obstacles", "cover"]},
    {"q": "What is a collection plan and how does IPB inform it?",
     "keywords": ["collection", "plan", "intelligence", "requirements", "assets"]},
    {"q": "How are threat models developed in IPB?",
     "keywords": ["threat", "model", "doctrine", "tactics", "template"]},
    {"q": "What is an avenue of approach?",
     "keywords": ["avenue", "approach", "route", "force", "objective"]},
    {"q": "How does IPB support unified land operations?",
     "keywords": ["unified", "operations", "commander", "planning", "intelligence"]},
]

def score_response(response, keywords):
    r    = response.lower()
    hits = sum(1 for kw in keywords if kw.lower() in r)
    return round(hits / len(keywords), 3) if keywords else 0.0

def run_inference(model, tokenizer, question):
    prompt = (
        f"<start_of_turn>user\n{EVAL_SYSTEM}\n\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, temperature=0.1,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

results = {'questions': [], 'summary': {}}

# ── Evaluate base model ───────────────────────────────────────
print('Loading BASE model...')
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_scores = []
for i, eq in enumerate(EVAL_QUESTIONS):
    resp  = run_inference(base_model, base_tok, eq['q'])
    score = score_response(resp, eq['keywords'])
    base_scores.append(score)
    results['questions'].append({'question': eq['q'], 'base_response': resp, 'base_score': score})
    print(f'  [{i+1:02d}/24] base score: {score:.3f}')

del base_model, base_tok
gc.collect(); torch.cuda.empty_cache()

# ── Evaluate fine-tuned model ─────────────────────────────────
print('\nLoading FINE-TUNED model...')
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

ft_scores = []
for i, eq in enumerate(EVAL_QUESTIONS):
    resp  = run_inference(ft_model, ft_tok, eq['q'])
    score = score_response(resp, eq['keywords'])
    ft_scores.append(score)
    results['questions'][i]['ft_response'] = resp
    results['questions'][i]['ft_score']    = score
    print(f'  [{i+1:02d}/24] ft score: {score:.3f}')

del ft_model, ft_tok
gc.collect(); torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
base_avg = round(float(np.mean(base_scores)), 4)
ft_avg   = round(float(np.mean(ft_scores)),   4)
improved = sum(1 for b, f in zip(base_scores, ft_scores) if f > b)

results['summary'] = {
    'base_avg':     base_avg,
    'ft_avg':       ft_avg,
    'improvement':  round((ft_avg - base_avg) / base_avg * 100, 1) if base_avg > 0 else 0,
    'improved_count': improved,
    'total_questions': len(EVAL_QUESTIONS),
    'dpo_candidates': sum(1 for s in ft_scores if s < 0.5),
}

os.makedirs(os.path.dirname(EVAL_PATH), exist_ok=True)
with open(EVAL_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '='*50)
print('  EVALUATION SUMMARY')
print('='*50)
print(f"  Base model avg score : {base_avg:.4f}")
print(f"  Fine-tuned avg score : {ft_avg:.4f}")
print(f"  Improvement          : +{results['summary']['improvement']}%")
print(f"  Examples improved    : {improved} / {len(EVAL_QUESTIONS)}")
print(f"  DPO candidates (<0.5): {results['summary']['dpo_candidates']}")
print('='*50)
print(f'\nResults saved → {EVAL_PATH}')

## Step 9 — Plot Results

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if 'DRIVE_BASE' not in dir():
    DRIVE_BASE = '/content/drive/MyDrive/atp-finetuning'
    EVAL_PATH  = f'{DRIVE_BASE}/eval/results/atp_gemma2_colab.json'
    CHART_PATH = f'{DRIVE_BASE}/eval/score_chart.png'

results = json.loads(open(EVAL_PATH).read())
qs      = results['questions']

labels      = [f"Q{i+1}" for i in range(len(qs))]
base_scores = [q['base_score'] for q in qs]
ft_scores   = [q['ft_score']   for q in qs]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width/2, base_scores, width, label='Base Gemma 2-2B', color='#6baed6', alpha=0.85)
ax.bar(x + width/2, ft_scores,   width, label='Fine-Tuned',       color='#2ca25f', alpha=0.85)

s = results['summary']
ax.set_title(
    f"ATP 2-01.3 Keyword Coverage: Base vs Fine-Tuned (Gemma 2-2B / Colab T4)\n"
    f"Base avg: {s['base_avg']:.4f}  |  Fine-tuned avg: {s['ft_avg']:.4f}  |  "
    f"Improvement: +{s['improvement']}%  |  {s['improved_count']}/{s['total_questions']} improved",
    fontsize=11,
)
ax.set_xlabel('Evaluation Question')
ax.set_ylabel('Keyword Coverage Score')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.1)
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='DPO threshold (0.5)')
ax.legend()
plt.tight_layout()

os.makedirs(os.path.dirname(CHART_PATH), exist_ok=True)
plt.savefig(CHART_PATH, dpi=150)
plt.show()
print(f'Chart saved → {CHART_PATH}')

## Step 10 — Export GGUF (Optional)

Exports the fine-tuned model to GGUF format for local deployment with Ollama.

In [ ]:
import gc, os
import torch
from unsloth import FastLanguageModel

if 'DRIVE_BASE' not in dir():
    DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
    ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
    GGUF_DIR     = f'{DRIVE_BASE}/burns'
    MODEL_NAME   = 'unsloth/gemma-2-2b-it'

ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH, max_seq_length=2048, dtype=None, load_in_4bit=True,
)

os.makedirs(GGUF_DIR, exist_ok=True)
ft_model.save_pretrained_gguf(
    f'{GGUF_DIR}/atp-gemma2-colab',
    ft_tok,
    quantization_method='q4_k_m',
)
print(f'GGUF exported → {GGUF_DIR}/atp-gemma2-colab/')
print('\nTo deploy locally with Ollama:')
print('  ollama create atp-doctrine -f Modelfile')
print('  ollama run atp-doctrine')